In [1]:
#write df in sqlite
import sqlite3
import pandas as pd

studies = pd.read_csv("../data/clean/studies.csv")
study_area = pd.read_csv("../data/clean/study_area.csv")
sites = pd.read_csv("../data/clean/sites.csv")
city_coords = pd.read_csv("../data/clean/city_coords.csv")

for name, df in [("studies", studies), ("study_area", study_area), ("sites", sites), ("city_coords", city_coords)]:
    print(name, df.shape, list(df.columns))

studies (1717, 10) ['nctId', 'studyType', 'phaseLabel', 'status', 'startDate', 'startType', 'startYear', 'startKind', 'sponsor', 'sponsorClass']
study_area (2849, 2) ['nctId', 'area']
sites (7524, 4) ['nctId', 'city', 'facility', 'siteStatus']
city_coords (74, 3) ['city', 'lat', 'lon']


In [2]:
#write df in database
conn = sqlite3.connect("../data/clean/trials.db")

studies.to_sql("studies", conn, if_exists="replace", index=False)
study_area.to_sql("study_area", conn, if_exists="replace", index=False)
sites.to_sql("sites", conn, if_exists="replace", index=False)
city_coords.to_sql("city_coords", conn, if_exists="replace", index=False)

74

In [3]:
#sql verification
conn = sqlite3.connect("../data/clean/trials.db")

for t in ["studies", "study_area", "sites", "city_coords"]:
    print(t, pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", conn)["n"][0])

print(pd.read_sql("SELECT COUNT(DISTINCT nctId) AS n FROM sites", conn))
print(pd.read_sql("SELECT COUNT(DISTINCT nctId) AS n FROM study_area", conn))

conn.close()

studies 1717
study_area 2849
sites 7524
city_coords 74
      n
0  1717
      n
0  1717


In [4]:
#alias table
alias_rules = {
    "Bristol-Myers Squibb": "bristol",
    "Merck & Co. (MSD)": r"merck sharp|merck & co",
    "Eli Lilly": "lilly",
    "Sanofi": "sanofi",
    "Novo Nordisk": "novo nordisk",
    "Pfizer": "pfizer",
    "Roche": r"roche|genentech",
    "Johnson & Johnson": r"janssen|johnson & johnson",
    "GSK": r"glaxosmithkline|gsk",
}

industry = studies.loc[studies["sponsorClass"] == "INDUSTRY", "sponsor"].drop_duplicates()

rows = []
for group, pattern in alias_rules.items():
    hit = industry[industry.str.contains(pattern, case=False, regex=True)]
    rows += [{"sponsor": s, "sponsorGroup": group} for s in hit]

sponsor_alias = pd.DataFrame(rows)
print(len(sponsor_alias), sponsor_alias["sponsor"].duplicated().sum())
print(sponsor_alias.sort_values(["sponsorGroup", "sponsor"]).to_string())

23 0
                                                                                 sponsor          sponsorGroup
0                                                                   Bristol-Myers Squibb  Bristol-Myers Squibb
2                                Juno Therapeutics, Inc., a Bristol-Myers Squibb Company  Bristol-Myers Squibb
1                              Karuna Therapeutics, Inc., a Bristol Myers Squibb company  Bristol-Myers Squibb
6                                                                  Eli Lilly and Company             Eli Lilly
7         Morphic Therapeutic, Inc. (A Wholly Owned Subsidiary of Eli Lilly and Company)             Eli Lilly
21                                                                       GlaxoSmithKline                   GSK
22                                                   Sierra Oncology LLC - a GSK company                   GSK
18                                                   Janssen Research & Development, LLC     Johnson & John

In [5]:
sponsor_alias.to_csv("../data/clean/sponsor_alias.csv", index=False)

conn = sqlite3.connect("../data/clean/trials.db")
sponsor_alias.to_sql("sponsor_alias", conn, if_exists="replace", index=False)
conn.close()